In [1]:
# packages

import numpy as np
import matplotlib as mpl
mpl.rcParams['axes.formatter.useoffset'] = False
import matplotlib.pyplot as plt
import scipy

from scipy.signal import sawtooth, square, savgol_filter
import pandas as pd
import glob as gl
import os
import cmath

from scipy.signal import sawtooth, square,find_peaks, savgol_filter
from scipy import spatial
#import lambdafit as lf
from scipy.interpolate import CubicSpline,interp1d
import h5py

from tqdm import tqdm as tqdm_terminal
from tqdm.notebook import trange, tqdm_notebook
from scipy.signal.windows import hann

from scipy.fft import fft, ifft, fftfreq
from copy import deepcopy
from scipy.interpolate import CubicSpline, interp1d
from scipy.optimize import curve_fit

# for pandas visual number display 

pd.set_option('display.precision', 6)
pd.set_option('display.float_format', '{:.10f}'.format)

# import ali_offline_demod.py 

import ali_offline_demod as aod

import time_chunking_functions as tcf

#### Now trying out time chunking functions but at larger scale... 

In [2]:
# data file

# LO: 4250
ts_LO4250_1 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4250.0_20240308061355_t_20240308062133/ts_toneinit_fcenter_4250.0_20240308061355_t_20240308062147.hd5'
ts_LO4250_1_fn = 'ts_toneinit_fcenter_4250.0_20240308061355_t_20240308062147.hd5'
t_xy_LO4250_1 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4250.0_20240308061355_t_20240308062133/beam_map_data_20240308062147.txt'
ts_LO4250_1_PATH = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4250.0_20240308061355_t_20240308062133'

ts_LO4250_2 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4250.0_20240307141217_t_20240307141951/ts_toneinit_fcenter_4250.0_20240307141217_t_20240307142007.hd5'
ts_LO4250_2_fn = 'ts_toneinit_fcenter_4250.0_20240307141217_t_20240307142007.hd5' 
t_xy_LO4250_2 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4250.0_20240307141217_t_20240307141951/beam_map_data_20240307142007.txt'
ts_LO4250_2_PATH = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4250.0_20240307141217_t_20240307141951'

# LO: 4750

ts_LO4750_1 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4750.0_20240309114324_t_20240309120414/ts_toneinit_fcenter_4750.0_20240309114324_t_20240309120429.hd5'
t_xy_LO4750_1 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4750.0_20240309114324_t_20240309120414/beam_map_data_20240309120429.txt'
ts_LO4750_1_PATH = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4750.0_20240309114324_t_20240309120414'

ts_LO4750_2 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4750.0_20240310035539_t_20240310040323/ts_toneinit_fcenter_4750.0_20240310035539_t_20240310040333.hd5'
t_xy_LO4750_2 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4750.0_20240310035539_t_20240310040323/beam_map_data_20240310040333.txt'
ts_LO4750_2_PATH = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4750.0_20240310035539_t_20240310040323'

ts_LO4750_3 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4750.0_20240310185738_t_20240310190410/ts_toneinit_fcenter_4750.0_20240310185738_t_20240310190423.hd5'
t_xy_LO4750_3 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4750.0_20240310185738_t_20240310190410/beam_map_data_20240310190423.txt'
ts_LO4750_3_PATH = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_4750.0_20240310185738_t_20240310190410'

# LO: 5250

ts_LO5250_1 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5250.0_20240311125818_t_20240311130516/ts_toneinit_fcenter_5250.0_20240311125818_t_20240311130530.hd5'
t_xy_LO5250_1 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5250.0_20240311125818_t_20240311130516/beam_map_data_20240311130530.txt'
ts_LO5250_1_PATH = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5250.0_20240311125818_t_20240311130516'

ts_LO5250_2 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5250.0_20240312134539_t_20240312135047/ts_toneinit_fcenter_5250.0_20240312134539_t_20240312135056.hd5'
t_xy_LO5250_2 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5250.0_20240312134539_t_20240312135047/beam_map_data_20240312135056.txt'
ts_LO5250_2_PATH = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5250.0_20240312134539_t_20240312135047'

ts_LO5250_3 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5250.0_20240313105637_t_20240313110245/ts_toneinit_fcenter_5250.0_20240313105637_t_20240313110254.hd5'
t_xy_LO5250_3 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5250.0_20240313105637_t_20240313110245/beam_map_data_20240313110254.txt'
ts_LO5250_3_PATH = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5250.0_20240313105637_t_20240313110245'

# LO: 5750

ts_LO5750_1 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5750.0_20240314100519_t_20240314101210/ts_toneinit_fcenter_5750.0_20240314100519_t_20240314101219.hd5'
t_xy_LO5750_1 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5750.0_20240314100519_t_20240314101210/beam_map_data_20240314101219.txt'
ts_LO5750_1_PATH = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5750.0_20240314100519_t_20240314101210'

ts_LO5750_2 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5750.0_20240318183441_t_20240318184103/ts_toneinit_fcenter_5750.0_20240318183441_t_20240318184325.hd5'
t_xy_LO5750_2 = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5750.0_20240318183441_t_20240318184103/beam_map_data_20240318184325.txt'
ts_LO5750_2_PATH = '/home/matt/ali_drive_mnt/beam_map_data/toneinit_fcenter_5750.0_20240318183441_t_20240318184103'


#### Now: process is to first get them in time-chunked state, then apply the modified full_demod_process function from the imported script

In [20]:
freq_list_ex_path = '/home/matt/ali_drive_mnt/tone_initializations/fcenter_4250.0_20240308061355/freq_list_lo_sweep_targeted_1_fcenter_4250.0_20240308061555.npy'
freq_list_ex = np.load(freq_list_ex_path)
freq_list_4250_1 = pd.DataFrame(freq_list_ex)

In [21]:
len(t_xy_LO4250_1)

127

In [22]:
testing_4250_1 = pd.read_csv(t_xy_LO4250_1, sep=',')
testing_4250_1

,start,end,x,y
0,1709907709.2978785038,1709907959.6177837849,0,0
1,1709907961.6202144623,1709907965.1014409065,10,0
2,1709907967.1036620140,1709907970.5852940083,20,0
3,1709907972.5858492851,1709907976.0646548271,30,0
4,1709907978.0658402443,1709907981.5459232330,40,0
...,...,...,...,...
5771,1709939586.7333009243,1709939590.2120680809,40,750
5772,1709939592.2142755985,1709939595.6955122948,30,750
5773,1709939597.6966049671,1709939601.1780261993,20,750
5774,1709939603.1804363728,1709939606.6595745087,10,750


In [6]:
freq_list_4250_1.loc[102:105]


#[4.22706738e+09+0.j 4.23421203e+09+0.j 4.23441903e+09+0.j
 #4.23421509e+09+0.j]

,0
102,4227067382.8125510216+0.0000000000j
103,4234212031.2500519753+0.0000000000j
104,4234419031.2500939369+0.0000000000j
105,4234215085.9375104904+0.0000000000j


In [7]:
t_4250_1, I_4250_1, Q_4250_1 = tcf.get_ts_chunks_time_indexing(ts_LO4250_1, t_xy_LO4250_1, chunk='some', chunk_start=102, chunk_stop=105, 
                                                               single_channel=None, time_chunk_index_start=1000, time_chunk_index_stop=1005)

4
[-1379 -1567 -1765 ... -1448 -1426 -1764]
[1.70990771e+09 1.70990796e+09 1.70990797e+09]
[1.70990796e+09 1.70990797e+09 1.70990797e+09]
True


In [8]:
def time_chunk_index_demod_plot(ts_filename, t_xy_file, chunk='some', chunk_start=None, chunk_stop=None, 
                                single_channel=None, time_chunk_index_start=0, time_chunk_index_stop=10,
                                ts_path='ts_path', ts_short_filename='ts_short_filename', 
                                tone_init_path='/home/matt/ali_drive_mnt/tone_initializations', 
                                plot_spacing=0.01):
    
    t, I, Q, t_xy_table = tcf.get_ts_chunks_time_indexing(ts_filename=ts_filename, t_xy_file=t_xy_file, chunk=chunk, chunk_start=chunk_start, 
                                                          chunk_stop=chunk_stop, single_channel=None, time_chunk_index_start=time_chunk_index_start,
                                                          time_chunk_index_stop=time_chunk_index_stop)
    
    chunks_demodulated = []

    for i in range(len(t)):
        t_chunked = t[i]
        i_chunked = np.array(I[i])
        q_chunked = np.array(Q[i])
        demodulated_chunk = tcf.modified_demod_process_for_tchunks(t_chunked, i_chunked, q_chunked, ts_file=ts_short_filename, f_sawtooth=15, method='fft', 
                                                                correct_phase_jumps=True, phase_jump_threshold=0.4, n=0, channels=chunk,
                                                                start_channel=chunk_start, stop_channel=chunk_stop, tone_init_path=tone_init_path,
                                                                ts_path=ts_path, display_mode='notebook')
        chunks_demodulated.append(demodulated_chunk)

        plt.figure()
        for j in range(len(demodulated_chunk['demod data'])):
            plt.plot(demodulated_chunk['demod t'], demodulated_chunk['demod data'][j] + plot_spacing*j, label=f'channel {chunk_start+j}')
            # plt.legend()
            plt.title(f'Time Chunk {time_chunk_index_start + i}')

    return chunks_demodulated, t_xy_table
    
    

In [12]:
%matplotlib qt

testing_full_function = time_chunk_index_demod_plot(ts_LO4250_2, t_xy_LO4250_2, chunk='some', chunk_start=210, chunk_stop=230,
                                                    time_chunk_index_start=2750, time_chunk_index_stop=2770, ts_path=ts_LO4250_2_PATH, 
                                                    ts_short_filename=ts_LO4250_2_fn, tone_init_path='/home/matt/ali_drive_mnt/tone_initializations', 
                                                    plot_spacing=0.1)

21
[2356 2422 2427 ... 2357 2299 2271]
[1.70985001e+09 1.70985026e+09 1.70985027e+09]
[1.70985026e+09 1.70985027e+09 1.70985027e+09]
True
using full_demod_process
4250.0
20240307141217
/home/matt/ali_drive_mnt/tone_initializations/fcenter_4250.0_20240307141217/
[4.45072461e+09+0.j 4.45211035e+09+0.j 4.45400405e+09+0.j
 4.45614258e+09+0.j 4.45847980e+09+0.j 4.46098159e+09+0.j
 4.46420831e+09+0.j 4.46682813e+09+0.j 4.46799316e+09+0.j
 4.46890346e+09+0.j 4.47056543e+09+0.j 4.47153530e+09+0.j
 4.47319857e+09+0.j 4.47535719e+09+0.j 4.47743192e+09+0.j
 4.47966341e+09+0.j 4.48259013e+09+0.j 4.48459877e+09+0.j
 4.48583027e+09+0.j 4.48813002e+09+0.j 4.49025816e+09+0.j]
num of channels: 21
num of tones: 21
looking for delay region
start = 4020292968.75
stop = 4030313476.5625
delay: 3.190635683471356e-08
3.190635683471356e-08
[4.45072461e+09+0.j 4.45211035e+09+0.j 4.45400405e+09+0.j
 4.45614258e+09+0.j 4.45847980e+09+0.j 4.46098159e+09+0.j
 4.46420831e+09+0.j 4.46682813e+09+0.j 4.46799316e+09+0.j

/home/matt/readout/lea_beammap_work/ali_hdf5_reading/time_chunking/ali_offline_demod.py:1393: RuntimeWarning: divide by zero encountered in log10
  y=np.append(y,20*np.log10(np.abs(test_sweep[1,:])))


  0%|          | 0/21 [00:00<?, ?it/s]

shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape sig: (977,)
len t: 977
len t_fr_start: 954
len sig_fr_start: 954
shape 

  0%|          | 0/21 [00:00<?, ?it/s]

...

shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape sig: (977,)
len t: 977
len t_fr_start: 959
len sig_fr_start: 959
shape 

  0%|          | 0/21 [00:00<?, ?it/s]

shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape sig: (976,)
len t: 976
len t_fr_start: 956
len sig_fr_start: 956
shape 

  0%|          | 0/21 [00:00<?, ?it/s]

shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape sig: (976,)
len t: 976
len t_fr_start: 962
len sig_fr_start: 962
shape 

In [5]:
xy_test = testing_full_function[1]
xy_test

,start,end,x,y
2000,1709918925.8174548149,1709918927.8198370934,240,260
2001,1709918931.3005313873,1709918933.3028566837,250,260
2002,1709918936.7840752602,1709918938.7862625122,260,260
2003,1709918942.2674095631,1709918944.2697718143,270,260
2004,1709918947.7516539097,1709918949.7540886402,280,260
2005,1709918953.2343800068,1709918955.2367277145,290,260
2006,1709918958.7178549767,1709918960.7201561928,300,260
2007,1709918964.2004890442,1709918966.2027668953,310,260
2008,1709918969.6837995052,1709918971.6861112118,320,260
2009,1709918975.1677112579,1709918977.1699721813,330,260


In [ ]:
chunks_demodulated=[]

for i in range(len(t_4250_1)):
    t_chunked = t_4250_1[i]
    i_chunked = np.array(I_4250_1[i])
    q_chunked = np.array(Q_4250_1[i])
    demodulated_chunk = tcf.modified_demod_process_for_tchunks(t_chunked, i_chunked, q_chunked, ts_file=ts_LO4250_1_fn, f_sawtooth=15, method='fft', 
                                                             correct_phase_jumps=True, phase_jump_threshold=0.4, n=0, channels='some',
                                                             start_channel=102, stop_channel=105, tone_init_path='/home/matt/ali_drive_mnt/tone_initializations',
                                                             ts_path=ts_LO4250_1_PATH, display_mode='notebook')
    
    chunks_demodulated.append(demodulated_chunk)

#### Where I ended 03/05/25 *** 

In [13]:
import time_chunking_functions as tcf

In [14]:
# just looking at what the data is like when I'm demodulating small part (not as small as 2 seconds) 

T, I, Q, ch, file = tcf.read_data(ts_LO4250_1, chunk='some', chunk_start=0, chunk_stop=20)

In [28]:
I[0,1]

-47

In [23]:
len(T)/32

486841.0

In [32]:
T_test_chunk = T[0:486841]
I_test_chunk = I[10, 0:486841]
Q_test_chunk = Q[10, 0:486841]

In [3]:
demod_testing = tcf.modified_demod_process_for_tchunks(t_test_chunk, )

(ts_LO4250_1_fn, f_sawtooth=15, method='fft', correct_phase_jumps=True, 
                                       phase_jump_threshold=0.4, n=0, channels='some', start_channel=0, stop_channel=2, 
                                       tone_init_path='/home/matt/ali_drive_mnt/tone_initializations', ts_path=ts_LO4250_1_PATH, 
                                       display_mode='notebook')

using full_demod_process
4250.0
20240308061355
/home/matt/ali_drive_mnt/tone_initializations/fcenter_4250.0_20240308061355/
[4.09184880e+09+0.j 4.09499819e+09+0.j 4.09772073e+09+0.j]


KeyboardInterrupt: 

In [30]:
len(T_test_chunk) == len(I_test_chunk)

True

In [31]:
plt.plot(T_test_chunk, I_test_chunk)